In [ ]:
# ── Cell 0: Setup ────────────────────────────────────────────────────────────
# Runtime → Change runtime type → CPU is fine (no training here)
# Requires: HF token with write access at https://hf.co/settings/tokens
import os, subprocess
from pathlib import Path

# ── Config — edit these ──────────────────────────────────────────────────────
HF_REPO   = 'your-hf-username/idiombert-joint-mbert'  # <-- set this
HF_TOKEN  = 'hf_...'                                   # <-- set this (write token)
# Drive path where training saved best_model/ (has pytorch_model.bin + task_heads.pt)
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Idiomator_Research/models/en_es_hi_te/joint_mbert/best_model'
# ─────────────────────────────────────────────────────────────────────────────

# Clone repo
REPO = '/content/Idiomator_Research'
if Path(REPO).exists():
    !cd {REPO} && git pull --ff-only
else:
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git {REPO}
%cd {REPO}
!pip install -q transformers torch huggingface_hub

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify model dir exists on Drive and has weights
model_path = Path(DRIVE_MODEL_DIR)
assert model_path.exists(), f'Drive model dir not found: {DRIVE_MODEL_DIR}'
files = list(model_path.iterdir())
print('Files in model dir:')
for f in sorted(files): print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

has_weights = any(f.name in ('pytorch_model.bin', 'model.safetensors') or f.name.startswith('model-') for f in files)
has_heads   = (model_path / 'task_heads.pt').exists()
assert has_weights, 'No backbone weights found — training may not have persisted to Drive'
assert has_heads,   'task_heads.pt missing'
print('✓ Backbone weights + task heads present')

In [ ]:
# ── Cell 1: Reconstruct model + sanity-check forward pass ────────────────────
import torch
from transformers import AutoModel, AutoTokenizer, PreTrainedModel, PretrainedConfig
from pathlib import Path

model_path = Path(DRIVE_MODEL_DIR)

# ── Rebuild JointIdiomModel exactly as in training/Train_Join.py ─────────────────────
class JointIdiomModel(torch.nn.Module):
    """
    Single mBERT encoder (google-bert/bert-base-multilingual-cased) with three
    task heads trained jointly for idiom detection:
      [CLS] → cls_head    (literal=0 / idiomatic=1)
      all tokens → start_head  (span-start logits)
      all tokens → end_head    (span-end logits)
    """
    def __init__(self, bert):
        super().__init__()
        self.bert       = bert
        hidden_size     = bert.config.hidden_size
        self.cls_head   = torch.nn.Linear(hidden_size, 2)
        self.start_head = torch.nn.Linear(hidden_size, 1)
        self.end_head   = torch.nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out        = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        seq        = out.last_hidden_state.float()
        cls        = seq[:, 0, :]
        cls_logits   = self.cls_head(cls)
        start_logits = self.start_head(seq).squeeze(-1)
        end_logits   = self.end_head(seq).squeeze(-1)
        mask = attention_mask.bool()
        start_logits = start_logits.masked_fill(~mask, float('-inf'))
        end_logits   = end_logits.masked_fill(~mask, float('-inf'))
        return cls_logits, start_logits, end_logits

# Load
print('Loading backbone from Drive...')
bert      = AutoModel.from_pretrained(str(model_path))
model     = JointIdiomModel(bert)
tokenizer = AutoTokenizer.from_pretrained(str(model_path))

heads = torch.load(model_path / 'task_heads.pt', map_location='cpu', weights_only=True)
model.cls_head.load_state_dict(heads['cls_head'])
model.start_head.load_state_dict(heads['start_head'])
model.end_head.load_state_dict(heads['end_head'])
model.eval()
print('✓ Model loaded')

# Sanity-check forward pass
test_sentence = 'He kicked the bucket last night .'
enc = tokenizer(test_sentence, return_tensors='pt')
with torch.no_grad():
    cls_logits, start_logits, end_logits = model(**enc)
pred_label = cls_logits.argmax(-1).item()  # 0=literal, 1=idiomatic
pred_start = start_logits.argmax(-1).item()
pred_end   = end_logits.argmax(-1).item()
tokens     = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
print(f'Test: "{test_sentence}"')
print(f'  label={pred_label} (1=idiomatic), span=[{pred_start}:{pred_end}] = {tokens[pred_start:pred_end+1]}')
print('✓ Forward pass OK')

In [ ]:
# ── Cell 2: Push to HuggingFace Hub ─────────────────────────────────────────
# Pushes: (a) fine-tuned backbone via push_to_hub, (b) task_heads.pt,
# (c) tokenizer, (d) model card with usage example.
# Load with trust_remote_code=True not required — backbone is standard BertModel.
import shutil, json
from huggingface_hub import HfApi, login
from pathlib import Path

login(token=HF_TOKEN)
api = HfApi()

# Create repo if needed
api.create_repo(repo_id=HF_REPO, exist_ok=True)
print(f'✓ Repo ready: https://huggingface.co/{HF_REPO}')

# Stage everything to a temp dir
STAGE = Path('/tmp/idiombert_push')
if STAGE.exists(): shutil.rmtree(STAGE)
STAGE.mkdir()

# Save fine-tuned backbone + tokenizer
model.bert.save_pretrained(str(STAGE))
tokenizer.save_pretrained(str(STAGE))

# Copy task heads
shutil.copy(model_path / 'task_heads.pt', STAGE / 'task_heads.pt')

# Write model card
readme = """
---
language:
- en
- es
- hi
- te
tags:
- idiom-detection
- multilingual
- span-extraction
base_model: google-bert/bert-base-multilingual-cased
---

# IdiomBERT — Joint mBERT for Multilingual Idiom Detection

Fine-tuned `google-bert/bert-base-multilingual-cased` for joint idiom detection across
English, Spanish, Hindi, and Telugu. One forward pass produces three outputs:
- **Classification**: literal (0) vs idiomatic (1)
- **Span start/end**: token indices of the idiomatic span

Trained on the MultiIdiom dataset (EN+ES+HI+TE split).

## Files
- `pytorch_model.bin` / `model.safetensors` — fine-tuned mBERT backbone
- `task_heads.pt` — three linear heads (`cls_head`, `start_head`, `end_head`)
- `tokenizer.*` — standard mBERT tokenizer

## Usage

```python
import torch
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download

REPO = "{hf_repo}"

backbone  = AutoModel.from_pretrained(REPO)
tokenizer = AutoTokenizer.from_pretrained(REPO)
heads     = torch.load(hf_hub_download(REPO, 'task_heads.pt'), map_location='cpu', weights_only=True)

# Attach heads
hidden = backbone.config.hidden_size  # 768
cls_head   = torch.nn.Linear(hidden, 2)
start_head = torch.nn.Linear(hidden, 1)
end_head   = torch.nn.Linear(hidden, 1)
cls_head.load_state_dict(heads['cls_head'])
start_head.load_state_dict(heads['start_head'])
end_head.load_state_dict(heads['end_head'])

backbone.eval(); cls_head.eval(); start_head.eval(); end_head.eval()

# Inference
enc = tokenizer("He kicked the bucket last night .", return_tensors='pt')
with torch.no_grad():
    seq     = backbone(**enc).last_hidden_state
    label   = cls_head(seq[:, 0, :]).argmax(-1).item()   # 0=literal, 1=idiomatic
    start   = start_head(seq).squeeze(-1).argmax(-1).item()
    end     = end_head(seq).squeeze(-1).argmax(-1).item()
print(label, start, end)
```
""".format(hf_repo=HF_REPO).strip()

(STAGE / 'README.md').write_text(readme)

# Verify staged files
print('Staged files:')
for f in sorted(STAGE.iterdir()): print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

# Push
print(f'\nPushing to {HF_REPO}...')
api.upload_folder(
    folder_path=str(STAGE),
    repo_id=HF_REPO,
    repo_type='model',
    commit_message='Add IdiomBERT joint mBERT EN+ES+HI+TE (backbone + task heads)',
)
print(f'✓ Done: https://huggingface.co/{HF_REPO}')

# Verify: try downloading task_heads.pt back
from huggingface_hub import hf_hub_download
downloaded = hf_hub_download(HF_REPO, 'task_heads.pt')
check = torch.load(downloaded, map_location='cpu', weights_only=True)
assert set(check.keys()) == {'cls_head', 'start_head', 'end_head'}
print('✓ task_heads.pt round-trip verified from HF')